## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model_groq = init_chat_model("groq:openai/gpt-oss-20b")
res = model_groq.invoke("Hello who are you?")
res

AIMessage(content='Hello! I’m ChatGPT, a conversational AI developed by OpenAI. I’m here to help answer questions, offer explanations, brainstorm ideas, or just chat about whatever’s on your mind. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user asks "Hello who are you?" We should answer politely.'}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 76, 'total_tokens': 154, 'completion_time': 0.083120972, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.004390499, 'prompt_tokens_details': None, 'queue_time': 0.395993687, 'total_time': 0.087511471}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a094ec-7568-71a1-87b0-f28f28d8eae6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 78, 'total_toke

In [4]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get Weather for a given location"""
    return f"The weather in {location} is rainy"

model_with_tool = model_groq.bind_tools([get_weather])

In [5]:
result = model_with_tool.invoke("What is the weather in Jharkhand?")
print(result)
for tool_calls in result.tool_calls:
    print(f"Tool: {tool_calls['name']}")
    print(f"Args: {tool_calls['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather function with location "Jharkhand".', 'tool_calls': [{'id': 'fc_b49c5eca-dece-4ec9-98a5-52a03d0725b1', 'function': {'arguments': '{"location":"Jharkhand"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 129, 'total_tokens': 169, 'completion_time': 0.040494158, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.006195732, 'prompt_tokens_details': None, 'queue_time': 0.316246907, 'total_time': 0.04668989}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_37c9245f64', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0950d-d045-7b50-af69-e3186e9a5fbf-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Jharkhand'}, 'id': 'fc_b49c5eca-dece-4ec9-98a5-52a03d0725b1', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'in

### Tool Execution Loop

In [6]:
# step 1 : Model generates tool calls
messages = [{"role": "user", "content": "What is the weather in Jharkhand?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# step 2 : Execute tool and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
# step 3 : Pass results back to the model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)


It’s rainy in Jharkhand.


In [7]:
messages

[{'role': 'user', 'content': 'What is the weather in Jharkhand?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in Jharkhand?" We can use the function get_weather with location "Jharkhand". Let\'s do that.', 'tool_calls': [{'id': 'fc_990804cd-c6da-4303-990e-3bd9c56165e8', 'function': {'arguments': '{"location":"Jharkhand"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 129, 'total_tokens': 187, 'completion_time': 0.065365655, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.029754269, 'prompt_tokens_details': None, 'queue_time': 0.35044981, 'total_time': 0.095119924}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09569-12d2-7b92-b9ce-82c5fcce0476-0', tool_calls=[{'name': 'get_weather',